# Explorar o lake no Colab

Importe escopos de uma base do DATASUS para um lake no seu Google Drive, veja as
linhas de um recorte com os rótulos do dicionário, o perfil de uma coluna e a citação
do que você escolheu. Os códigos ficam como o DATASUS publicou; o rótulo vem do
dicionário, ao lado (`sexo_rotulo`).

Cada seção tem campos de formulário: mude o valor e rode a célula. Rode as seções em
ordem na primeira vez (**Ambiente de execução → Executar tudo** também funciona). Com o
recorte inicial (SIM de Roraima, 2022), o download tem menos de 1 MB.

O lake fica no Drive e continua lá depois que o ambiente do Colab é desligado: na
próxima vez, desmarque `IMPORTAR` para explorar sem consultar o DATASUS.

> **Trabalho em construção.** Confira as contagens com o DATASUS e os rótulos com a
> fonte antes de publicar: [como conferir](https://raphaelfh.github.io/omnisus/dicionario/).

## 1. Instalar

Instala com o `uv`, que já vem no Colab, a versão do PyPI fixada abaixo.

In [ ]:
!uv pip install -q "omnisus[colab]==0.2.3"

## 2. Onde fica o lake

Com `USAR_DRIVE`, o lake fica em `PASTA_NO_DRIVE`, dentro de *Meu Drive*: o
[notebook do Colab](https://github.com/raphaelfh/omnisus/blob/main/notebooks/colab.ipynb)
grava em `omnisus`, e este notebook lê e completa o mesmo lake. Na primeira vez, o
Colab pede para acessar o seu Drive. Sem o Drive, o lake fica em `/content/data/raw` e
some quando o ambiente é desligado.

Grave no lake de um notebook por vez: duas sessões gravando juntas não são protegidas.

In [ ]:
import json

import polars as pl

import omnisus as sus

# Até 25 linhas e 80 caracteres por célula em cada tabela.
pl.Config.set_tbl_rows(25).set_fmt_str_lengths(80)

USAR_DRIVE = True  # @param {type:"boolean"}
PASTA_NO_DRIVE = "omnisus"  # @param {type:"string"}

if USAR_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    sus.set_lake_dir(f"/content/drive/MyDrive/{PASTA_NO_DRIVE}")


def valores(texto, tipo=str):
    """`"2022, 2023"` ou `"2022 2023"` como lista; um campo vazio é a lista vazia."""
    return [tipo(v) for v in texto.replace(",", " ").split()]

## 3. Importar

`BASE` é o nome de uma base: a página
[Bases e argumentos](https://raphaelfh.github.io/omnisus/datasets/) lista todas. Escreva
os anos, as UFs e os meses separados por vírgula. Uma base nacional (SINAN, subconjuntos
do SIM) ignora `UFS`; uma base anual ignora `MESES`.

O que o lake já tem do mesmo arquivo não é baixado de novo. Ligue `SUBSTITUIR` só
quando o motivo de um escopo pedir `replace`: o DATASUS publicou o arquivo de novo, ou
outra versão do omnisus o importou. O relatório diz o que houve com cada escopo; para
continuar depois de uma falha, rode a célula de novo.

In [ ]:
BASE = "sim_obitos"  # @param {type:"string"}
ANOS = "2022"  # @param {type:"string"}
UFS = "RR"  # @param {type:"string"}
MESES = "1"  # @param {type:"string"}
SUBSTITUIR = False  # @param {type:"boolean"}
IMPORTAR = True  # @param {type:"boolean"}

if IMPORTAR:
    base = sus.resolve(BASE)
    escopos_novos = sus.scopes_for(
        BASE,
        years=valores(ANOS, int),
        ufs=[u.upper() for u in valores(UFS)] if base.geography == "state" else None,
        months=valores(MESES, int) if base.monthly else None,
    )
    situacoes = {
        None: "importado",
        "unchanged": "já estava no lake",
        "not_listed": "o DATASUS não publica",
        "outside_coverage": "fora da cobertura da base",
        "fetch_failed": "falhou ao baixar",
        "ingest_failed": "falhou ao gravar",
        "rolled_back": "desfeito: outro escopo da parte falhou",
    }
    importacao = sus.import_dataset(
        BASE, scopes=escopos_novos, policy="replace" if SUBSTITUIR else "skip_same"
    )
    relatorio = pl.DataFrame(
        [(str(o.scope), situacoes[o.code], o.reason) for o in importacao.outcomes],
        schema=dict.fromkeys(["escopo", "situacao", "motivo"], pl.String),
        orient="row",
    )
    display(relatorio)

## 4. O que o lake tem

Um escopo por linha, lido do catálogo, sem rede: a base, a UF, o ano e o mês, o
diretório do servidor (`final` ou `prelim`), quantas linhas foram gravadas e quando. As
seções seguintes leem o lake neste snapshot.

In [ ]:
with sus.LakeReader() as leitor:
    snapshot = sus.latest_snapshot_id(leitor)
with sus.LakeReader(snapshot_id=snapshot) as leitor:
    publicacoes = [p for p in leitor.publications() if p["active"] and p["scope"] is not None]

no_lake = pl.DataFrame(
    [
        {
            "base": p["dataset"],
            "uf": p["scope"].uf,
            "ano": p["scope"].ano,
            "mes": p["scope"].mes,
            "diretorio": p["release"],
            "linhas": p["rows"],
            "importado_em": p["published_at"],
        }
        for p in publicacoes
    ],
    schema_overrides={"uf": pl.String, "mes": pl.Int64},
).sort("base", "ano", "mes", "uf", nulls_last=True)
no_lake

## 5. Ver as linhas

Escolha uma base que o lake tem e, se quiser, as UFs, os anos e os meses; um campo
vazio deixa passar todos. A contagem é de todos os escopos escolhidos; a tabela mostra
até `LINHAS` delas, porque cada linha ocupa a memória do Colab. Com `ROTULOS`, cada
código que o dicionário conhece ganha a coluna `<coluna>_rotulo`.

As categorias harmonizadas (`idade_anos_completos`, `sexo_categoria`...) só aparecem
quando todos os escopos vêm de fontes validadas; a célula diz o motivo quando faltam.
Veja [consumo dos metadados](https://raphaelfh.github.io/omnisus/dicionario/consumo/).

In [ ]:
BASE_EXPLORAR = "sim_obitos"  # @param {type:"string"}
UFS_EXPLORAR = ""  # @param {type:"string"}
ANOS_EXPLORAR = ""  # @param {type:"string"}
MESES_EXPLORAR = ""  # @param {type:"string"}
LINHAS = 10000  # @param {type:"integer"}
ROTULOS = True  # @param {type:"boolean"}

ufs = [u.upper() for u in valores(UFS_EXPLORAR)]
anos = valores(ANOS_EXPLORAR, int)
meses = valores(MESES_EXPLORAR, int)


def escolhido(escopo):
    """Uma base nacional não tem UF e uma anual não tem mês: esses filtros não as tiram."""
    return (
        (not ufs or escopo.uf is None or escopo.uf in ufs)
        and (not anos or escopo.ano in anos)
        and (not meses or escopo.mes is None or escopo.mes in meses)
    )


escolhidas = [p for p in publicacoes if p["dataset"] == BASE_EXPLORAR and escolhido(p["scope"])]
if not escolhidas:
    raise ValueError(
        f"O lake não tem escopos de {BASE_EXPLORAR} nesse recorte: veja a tabela da seção 4."
    )
total = sum(p["rows"] for p in escolhidas)

# Cada publicação guarda em `scope_json` as colunas e os valores que gravou (`uf`, `ano`,
# `mes`; `_source_ano` numa base nacional): o WHERE seleciona exatamente esses escopos.
campos = [json.loads(p["scope_json"]) for p in escolhidas]
chaves = list(campos[0])
colunas = ", ".join(f'"{c}"' for c in chaves)
listas = ", ".join("unnest(?)" for _ in chaves)
onde = f"({colunas}) IN (SELECT {listas})"
parametros = [[e[c] for e in campos] for c in chaves]

with sus.LakeReader(snapshot_id=snapshot) as leitor:
    con = leitor.connect()
    esquema = {n: t for n, t, *_ in con.execute(f'DESCRIBE lake."{BASE_EXPLORAR}"').fetchall()}
    projecao = sus.analytical_projection(
        BASE_EXPLORAR,
        observed_schema=esquema,
        scopes=[sus.SourceContext.from_publication(p) for p in escolhidas],
    )
    categorias = "".join(f', {c.expression} AS "{c.name}"' for c in projecao.columns)
    linhas = con.execute(
        f'SELECT *{categorias} FROM lake."{BASE_EXPLORAR}" WHERE {onde} LIMIT {int(LINHAS)}',
        parametros,
    ).pl()
if ROTULOS:
    linhas = sus.label(BASE_EXPLORAR, linhas)

print(f"{total} linhas nos escopos escolhidos; a tabela mostra {linhas.height}.")
for u in projecao.unavailable:
    if u.reason != "unsupported_dataset":
        print(f"Categoria harmonizada fora: {u.field} ({u.reason}).")
linhas

## 6. Perfil de uma coluna

Quantas linhas tem cada código de `COLUNA`, em todos os escopos escolhidos na seção 5
(não só nas da tabela), com o rótulo do dicionário. Troque pelo nome de uma coluna da
base: `sexo` é do SIM.

In [ ]:
COLUNA = "sexo"  # @param {type:"string"}

with sus.LakeReader(snapshot_id=snapshot) as leitor:
    perfil = (
        leitor.connect()
        .execute(
            f'SELECT "{COLUNA}", count(*) AS registros, '
            "round(100 * count(*) / sum(count(*)) OVER (), 2) AS pct "
            f'FROM lake."{BASE_EXPLORAR}" WHERE {onde} '
            "GROUP BY ALL ORDER BY registros DESC LIMIT 1000",
            parametros,
        )
        .pl()
    )
perfil = sus.label(BASE_EXPLORAR, perfil)
perfil

## 7. Citar

A citação nomeia o arquivo do servidor, o SHA-256 e o snapshot do lake de cada escopo
escolhido na seção 5. Guarde-a junto com o resultado; veja
[Reprodutibilidade](https://raphaelfh.github.io/omnisus/pesquisa/reprodutibilidade/).

In [ ]:
citacao = sus.citation_from_publications(
    escolhidas, snapshot_id=snapshot, dataset=BASE_EXPLORAR
).text
print(citacao)

## Antes de fechar

Se importou com o Drive, rode `drive.flush_and_unmount()` para todos os arquivos
chegarem lá. Os [notebooks de cada base](https://github.com/raphaelfh/omnisus/tree/main/notebooks)
aprofundam cada sistema, e o [guia do pesquisador](https://raphaelfh.github.io/omnisus/pesquisa/)
explica as escolhas.